# Project 2

## Configuration

In [1]:
import time
import os
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import (
    StructType, StructField, IntegerType, DoubleType, TimestampType
)
# Packages are loaded via PYSPARK_SUBMIT_ARGS set in compose.yml.
# pyspark-notebook:2025-12-31 ships Spark 4.1.0 — print spark.version to confirm.

spark = (
    SparkSession.builder
    .appName("project2")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")

    # ── Iceberg ──────────────────────────────────────────────────────────────
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    # Catalog named 'lakehouse' — use it as: lakehouse.<database>.<table>
    .config("spark.sql.catalog.lakehouse",
            "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lakehouse.type",      "rest")
    .config("spark.sql.catalog.lakehouse.uri",       "http://iceberg-rest:8181")
    .config("spark.sql.catalog.lakehouse.warehouse", "s3://warehouse/")
    # S3FileIO writes data files directly to MinIO
    .config("spark.sql.catalog.lakehouse.io-impl",
            "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.lakehouse.s3.endpoint",          "http://minio:9000")
    .config("spark.sql.catalog.lakehouse.s3.path-style-access", "true")
    .config("spark.sql.catalog.lakehouse.s3.access-key-id",     os.environ["AWS_ACCESS_KEY_ID"])
    .config("spark.sql.catalog.lakehouse.s3.secret-access-key", os.environ["AWS_SECRET_ACCESS_KEY"])
    .config("spark.sql.catalog.lakehouse.s3.region", "us-east-1")

    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version}   catalog: lakehouse")

# ── Create your database once ──────────────────────────────────────────────
spark.sql("CREATE DATABASE IF NOT EXISTS lakehouse.taxi")

Spark 4.1.0   catalog: lakehouse


DataFrame[]

In [2]:
BOOTSTRAP = "kafka:9092"
TOPIC     = "taxi-trips"

raw_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("subscribe", TOPIC)
    .option("startingOffsets", "earliest")
    .load()
)

In [3]:
# Cleanup
# import shutil
# shutil.rmtree("/tmp/chk-bronze", ignore_errors=True)
# spark.sql("TRUNCATE TABLE lakehouse.taxi.silver_trips")
# spark.sql("TRUNCATE TABLE lakehouse.taxi.bronze_trips")
# spark.sql("TRUNCATE TABLE lakehouse.taxi.gold_route_stats")

## Bronze layer

In [4]:
spark.sql("""
CREATE TABLE IF NOT EXISTS lakehouse.taxi.bronze_trips (
    key STRING,
    value STRING,
    topic STRING,
    partition INT,
    offset BIGINT,
    timestamp TIMESTAMP
)
USING iceberg
""")

bronze_source = (
    raw_stream
    .select(
        F.col("key").cast("string"),
        F.col("value").cast("string"),
        F.col("topic"),
        F.col("partition"),
        F.col("offset"),
        F.col("timestamp")
    )
)

bronze_query = (
    bronze_source.writeStream
    .format("iceberg")
    .outputMode("append")
    .option("checkpointLocation", "/tmp/chk-bronze")
    .trigger(processingTime="5 seconds")
    .toTable("lakehouse.taxi.bronze_trips")
)

time.sleep(12)

count_before = spark.read.table("lakehouse.taxi.bronze_trips").count()
print(f"Row count before restart: {count_before}")

bronze_query.stop()
print("Query stopped.")

# Restart the query using the SAME checkpoint directory.
# Even though startingOffsets is still 'earliest', Spark ignores that setting
# and resumes from the committed offsets in the checkpoint.

bronze_query2 = (
    bronze_source.writeStream
    .format("iceberg")
    .outputMode("append")
    .option("checkpointLocation", "/tmp/chk-bronze")
    .trigger(processingTime="5 seconds")
    .toTable("lakehouse.taxi.bronze_trips")
)

time.sleep(12)   # wait for two triggers

count_after = spark.read.table("lakehouse.taxi.bronze_trips").count()
print(f"Row count before restart : {count_before}")
print(f"Row count after restart  : {count_after}")
print()
if count_after == count_before:
    print("Counts are equal — the checkpoint prevented re-ingestion of already-processed messages.")
else:
    print(f"Counts differ by {count_after - count_before} — "
          "those are new messages produced between the two runs.")

bronze_query2.stop()

spark.sql("""SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT value) AS unique_events
FROM lakehouse.taxi.bronze_trips;""").show()

Row count before restart: 2362
Query stopped.
Row count before restart : 2362
Row count after restart  : 2411

Counts differ by 49 — those are new messages produced between the two runs.
+----------+-------------+
|total_rows|unique_events|
+----------+-------------+
|      2411|         2411|
+----------+-------------+



In [5]:
spark.sql("""
SELECT count(*) FROM lakehouse.taxi.bronze_trips;
""").show()

spark.sql("""
SELECT * FROM lakehouse.taxi.bronze_trips LIMIT 3;
""").show()
# The checkpoint location is configured in the bronze writeStream query: 
#.option("checkpointLocation", "/tmp/chk-bronze")

+--------+
|count(1)|
+--------+
|    2411|
+--------+

+---+--------------------+----------+---------+------+--------------------+
|key|               value|     topic|partition|offset|           timestamp|
+---+--------------------+----------+---------+------+--------------------+
|  1|{"VendorID": 1, "...|taxi-trips|        0|   928|2026-04-03 22:26:...|
|  1|{"VendorID": 1, "...|taxi-trips|        0|   929|2026-04-03 22:26:...|
|  1|{"VendorID": 1, "...|taxi-trips|        0|   930|2026-04-03 22:26:...|
+---+--------------------+----------+---------+------+--------------------+



# Silver layer

In [6]:
zones = spark.read.parquet("data/taxi_zone_lookup.parquet")

spark.sql("""
CREATE TABLE IF NOT EXISTS lakehouse.taxi.silver_trips (
    VendorID             INT,
    tpep_pickup_datetime TIMESTAMP,
    tpep_dropoff_datetime TIMESTAMP,
    passenger_count      INT,
    trip_distance        DOUBLE,
    PULocationID         INT,
    DOLocationID         INT,
    fare_amount          DOUBLE,
    total_amount         DOUBLE,
    pickup_borough       STRING,
    pickup_zone          STRING,
    pickup_service_zone  STRING,
    dropoff_borough      STRING,
    dropoff_zone         STRING,
    dropoff_service_zone STRING
)
USING ICEBERG
PARTITIONED BY (days(tpep_pickup_datetime))
""")

schema = StructType([
    StructField("VendorID",              IntegerType()),
    StructField("tpep_pickup_datetime",  TimestampType()),
    StructField("tpep_dropoff_datetime", TimestampType()),
    StructField("passenger_count",       IntegerType()),
    StructField("trip_distance",         DoubleType()),
    StructField("PULocationID",          IntegerType()),
    StructField("DOLocationID",          IntegerType()),
    StructField("fare_amount",           DoubleType()),
    StructField("total_amount",          DoubleType()),
])

pu = zones.alias("pu")
do = zones.alias("do")

parsed = (
    spark.table("lakehouse.taxi.bronze_trips")
    .withColumn("json", F.from_json(F.col("value").cast("string"), schema))
    .select("json.*")
)

transformed = (
    parsed
    .select(
        F.col("VendorID").cast("int"),
        F.col("tpep_pickup_datetime"),
        F.col("tpep_dropoff_datetime"),
        F.when(
            (F.col("passenger_count").isNull()) | (F.col("passenger_count") <= 0), 1
        ).otherwise(F.col("passenger_count")).cast("int").alias("passenger_count"),
        F.col("trip_distance").cast("double"),
        F.col("PULocationID").cast("int"),
        F.col("DOLocationID").cast("int"),
        F.col("fare_amount").cast("double"),
        F.col("total_amount").cast("double"),
    )
    .dropna(subset=[
        "tpep_pickup_datetime", "tpep_dropoff_datetime",
        "PULocationID", "DOLocationID"
    ])
    .filter(F.col("tpep_dropoff_datetime") > F.col("tpep_pickup_datetime"))
    .filter(F.col("trip_distance") > 0)
    .filter((F.col("passenger_count") > 0) & (F.col("passenger_count") < 10))
    .filter(F.col("fare_amount") >= 0)
)

enriched = (
    transformed
    .join(F.broadcast(pu), transformed.PULocationID == F.col("pu.LocationID"), "left")
    .join(F.broadcast(do), transformed.DOLocationID == F.col("do.LocationID"), "left")
    .select(
        transformed["*"],
        F.col("pu.Borough").alias("pickup_borough"),
        F.col("pu.Zone").alias("pickup_zone"),
        F.col("pu.service_zone").alias("pickup_service_zone"),
        F.col("do.Borough").alias("dropoff_borough"),
        F.col("do.Zone").alias("dropoff_zone"),
        F.col("do.service_zone").alias("dropoff_service_zone"),
    )
    .filter(
        F.col("pickup_zone").isNotNull() &
        F.col("dropoff_zone").isNotNull()
    )
)

# overwritePartitions instead of append
print("First run bronze -> silver")
enriched.writeTo("lakehouse.taxi.silver_trips").overwritePartitions()

bronze_count = spark.table("lakehouse.taxi.bronze_trips").count()
silver_count = spark.table("lakehouse.taxi.silver_trips").count()
print(f"Bronze rows : {bronze_count}")
print(f"Silver rows : {silver_count}")
print(f"Drop rate   : {1 - silver_count/bronze_count:.1%}  (filtered/invalid records)\n")

# rerun write
enriched.writeTo("lakehouse.taxi.silver_trips").overwritePartitions()

bronze_count = spark.table("lakehouse.taxi.bronze_trips").count()
silver_count = spark.table("lakehouse.taxi.silver_trips").count()
print("After rerunning writeTo silver_trips")
print(f"Bronze rows : {bronze_count}")
print(f"Silver rows : {silver_count}")
print(f"Drop rate   : {1 - silver_count/bronze_count:.1%}  (filtered/invalid records)")

First run bronze -> silver
Bronze rows : 2411
Silver rows : 2354
Drop rate   : 2.4%  (filtered/invalid records)

After rerunning writeTo silver_trips
Bronze rows : 2411
Silver rows : 2354
Drop rate   : 2.4%  (filtered/invalid records)


In [7]:
silver_check = spark.read.table("lakehouse.taxi.silver_trips")
silver_check.show(5, truncate=False)
print(f"Silver count: {silver_check.count()}\n")

print("Null counts per column")
silver_check.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in silver_check.columns
]).show()

print("Trip distribution by borough")
silver_check.groupBy("pickup_borough").count().show()

print("Trip duration statistics")
silver_check.select(
    (F.col("tpep_dropoff_datetime").cast("long") - 
     F.col("tpep_pickup_datetime").cast("long")).alias("duration_sec")
).describe().show()

print("Total vs unique")
spark.sql("""SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT *) AS unique_rows
FROM lakehouse.taxi.silver_trips;""").show()

+--------+--------------------+---------------------+---------------+-------------+------------+------------+-----------+------------+--------------+-----------------------------+-------------------+---------------+-----------------+--------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|PULocationID|DOLocationID|fare_amount|total_amount|pickup_borough|pickup_zone                  |pickup_service_zone|dropoff_borough|dropoff_zone     |dropoff_service_zone|
+--------+--------------------+---------------------+---------------+-------------+------------+------------+-----------+------------+--------------+-----------------------------+-------------------+---------------+-----------------+--------------------+
|2       |2024-12-31 23:30:03 |2024-12-31 23:43:02  |1              |3.0          |246         |13          |16.3       |26.62       |Manhattan     |West Chelsea/Hudson Yards    |Yellow Zone        |Manhattan      |Battery Park City|Ye

# Gold layer

In [8]:

spark.sql("""
CREATE TABLE IF NOT EXISTS lakehouse.taxi.gold_route_stats (
    pickup_borough    STRING,
    dropoff_borough   STRING,
    trip_count        BIGINT,
    total_revenue     DOUBLE,
    avg_distance      DOUBLE,
    revenue_per_mile  DOUBLE
)
USING iceberg
PARTITIONED BY (pickup_borough)
""")


DataFrame[]

In [9]:
def update_gold_route_stats():
    """
    Reads from Silver, calculates aggregates, and overwrites the Gold table.
    """

    gold_route_df = (
        spark.table("lakehouse.taxi.silver_trips")
        .groupBy("pickup_borough", "dropoff_borough")
        .agg(
            F.count("*").alias("trip_count"),
            F.round(F.sum("total_amount"), 2).alias("total_revenue"),
            F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
            F.round(
                F.sum("total_amount") / F.when(F.sum("trip_distance") > 0, F.sum("trip_distance")).otherwise(1), 
                2
            ).alias("revenue_per_mile")
        )
    )

    gold_route_df.writeTo("lakehouse.taxi.gold_route_stats").overwritePartitions()


#### Verify, that restart does not produce duplicates

In [10]:
# initial run
update_gold_route_stats()
count_initial = spark.table("lakehouse.taxi.gold_route_stats").count()
print(f"Row count after initial run: {count_initial}")
print(f"Data after initial run: {count_initial}")
spark.table("lakehouse.taxi.gold_route_stats").sort(F.desc("trip_count")).show()

# duplicate run
update_gold_route_stats()
count_after_restart = spark.table("lakehouse.taxi.gold_route_stats").count()


print(f"Row count second run  : {count_after_restart}")
print(f"Data after second run: {count_initial}")
spark.table("lakehouse.taxi.gold_route_stats").sort(F.desc("trip_count")).show()
    

Row count after initial run: 17
Data after initial run: 17
+--------------+---------------+----------+-------------+------------+----------------+
|pickup_borough|dropoff_borough|trip_count|total_revenue|avg_distance|revenue_per_mile|
+--------------+---------------+----------+-------------+------------+----------------+
|     Manhattan|      Manhattan|      2125|     47517.96|         2.1|           10.64|
|     Manhattan|       Brooklyn|        69|      3023.08|        6.25|            7.01|
|     Manhattan|         Queens|        41|      1927.53|        7.56|            6.22|
|        Queens|      Manhattan|        24|      1589.89|        10.7|            6.19|
|        Queens|       Brooklyn|        22|      1299.34|       11.07|            5.33|
|        Queens|         Queens|        18|       581.23|         4.8|            6.73|
|     Manhattan|          Bronx|        12|       591.17|        9.83|            5.01|
|      Brooklyn|       Brooklyn|         7|       196.75|    

In [11]:
gold_audit_df = spark.sql("""
    SELECT 
        snapshot_id,
        committed_at,
        operation,
        summary['replace-partitions'] AS is_overwrite,
        summary['added-records']      AS rows_added,
        summary['deleted-records']    AS rows_deleted,
        summary['total-records']      AS total_table_rows,
        summary['added-data-files']   AS files_added,
        summary['deleted-data-files'] AS files_deleted
    FROM lakehouse.taxi.gold_route_stats.snapshots
    ORDER BY committed_at DESC
""")

gold_audit_df.show(truncate=False)

+-------------------+-----------------------+---------+------------+----------+------------+----------------+-----------+-------------+
|snapshot_id        |committed_at           |operation|is_overwrite|rows_added|rows_deleted|total_table_rows|files_added|files_deleted|
+-------------------+-----------------------+---------+------------+----------+------------+----------------+-----------+-------------+
|8493017191577700046|2026-04-03 22:26:37.475|overwrite|true        |17        |17          |17              |4          |4            |
|4371350720555103131|2026-04-03 22:26:36.717|overwrite|true        |17        |10          |17              |4          |4            |
|5653346914531566062|2026-04-03 22:19:10.181|overwrite|true        |10        |10          |10              |4          |4            |
|3178187457043086020|2026-04-03 22:19:09.578|overwrite|true        |10        |NULL        |10              |4          |NULL         |
+-------------------+-----------------------+---